# 02_Cohort_Creation

## Objective
The goal of this notebook is to create the final patient cohort for the MRP project.

This cohort will be built by:
- merging `PATIENTS`, `ADMISSIONS`, and `ICUSTAYS`
- extracting patient age
- keeping only adult patients
- selecting the first ICU stay only
- creating a sepsis label using ICD-9 diagnosis codes from `DIAGNOSES_ICD`

The final output will be a cleaned cohort dataset saved for the next stage of the project.

In [1]:
# Code Starter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

DATA_PATH = "../data/raw/"
PROCESSED_PATH = "../data/processed/"
OUTPUT_PATH = "../outputs/"

### Load Core Tables

In [5]:
admissions = pd.read_csv(DATA_PATH + "ADMISSIONS.csv")
patients = pd.read_csv(DATA_PATH + "PATIENTS.csv")
icustays = pd.read_csv(DATA_PATH + "ICUSTAYS.csv")
diagnoses = pd.read_csv(DATA_PATH + "DIAGNOSES_ICD.csv")

print("ADMISSIONS:", admissions.shape)
print("PATIENTS:", patients.shape)
print("ICUSTAYS:", icustays.shape)
print("DIAGNOSES_ICD:", diagnoses.shape)

ADMISSIONS: (58976, 19)
PATIENTS: (46520, 8)
ICUSTAYS: (61532, 12)
DIAGNOSES_ICD: (651047, 5)


## Step 1: Convert date columns
The relevant date columns are converted to datetime format so that age and time-based filtering can be performed correctly.

In [8]:
# Convert date columns to datetime
admissions["ADMITTIME"] = pd.to_datetime(admissions["ADMITTIME"])
admissions["DISCHTIME"] = pd.to_datetime(admissions["DISCHTIME"])
admissions["DEATHTIME"] = pd.to_datetime(admissions["DEATHTIME"], errors="coerce")

patients["DOB"] = pd.to_datetime(patients["DOB"])
patients["DOD"] = pd.to_datetime(patients["DOD"], errors="coerce")

icustays["INTIME"] = pd.to_datetime(icustays["INTIME"])
icustays["OUTTIME"] = pd.to_datetime(icustays["OUTTIME"], errors="coerce")

## Step 2: Merge PATIENTS, ADMISSIONS, and ICUSTAYS
The three tables are merged to create a single dataframe containing patient demographics, hospital admission information, and ICU stay details.

In [11]:
# Merge tables
cohort = icustays.merge(admissions, on=["SUBJECT_ID", "HADM_ID"], how="inner")
cohort = cohort.merge(patients, on="SUBJECT_ID", how="inner")

print("Merged cohort shape:", cohort.shape)
cohort.head()

Merged cohort shape: (61532, 36)


,ROW_ID_x,SUBJECT_ID,HADM_ID,ICUSTAY_ID,DBSOURCE,FIRST_CAREUNIT,LAST_CAREUNIT,FIRST_WARDID,LAST_WARDID,INTIME,OUTTIME,LOS,ROW_ID_y,ADMITTIME,DISCHTIME,DEATHTIME,ADMISSION_TYPE,ADMISSION_LOCATION,DISCHARGE_LOCATION,INSURANCE,LANGUAGE,RELIGION,MARITAL_STATUS,ETHNICITY,EDREGTIME,EDOUTTIME,DIAGNOSIS,HOSPITAL_EXPIRE_FLAG,HAS_CHARTEVENTS_DATA,ROW_ID,GENDER,DOB,DOD,DOD_HOSP,DOD_SSN,EXPIRE_FLAG
0,365,268,110404,280836,carevue,MICU,MICU,52,52,2198-02-14 23:27:38,2198-02-18 05:26:11,3.2490,344,2198-02-11 13:40:00,2198-02-18 03:55:00,2198-02-18 03:55:00,EMERGENCY,EMERGENCY ROOM ADMIT,DEAD/EXPIRED,Medicare,NaN,CATHOLIC,SEPARATED,HISPANIC OR LATINO,2198-02-11 09:41:00,2198-02-11 15:18:00,DYSPNEA,1,1,251,F,2132-02-21,2198-02-18,2198-02-18 00:00:00,NaN,1
1,366,269,106296,206613,carevue,MICU,MICU,52,52,2170-11-05 11:05:29,2170-11-08 17:46:57,3.2788,345,2170-11-05 11:04:00,2170-11-27 18:00:00,NaT,EMERGENCY,EMERGENCY ROOM ADMIT,HOME HEALTH CARE,Medicaid,NaN,UNOBTAINABLE,SINGLE,WHITE,2170-11-05 07:22:00,2170-11-05 12:15:00,SEPSIS;PILONIDAL ABSCESS,0,1,252,M,2130-09-30,NaT,NaN,NaN,0
2,367,270,188028,220345,carevue,CCU,CCU,57,57,2128-06-24 15:05:20,2128-06-27 12:32:29,2.8939,346,2128-06-23 18:26:00,2128-06-27 12:31:00,NaT,ELECTIVE,PHYS REFERRAL/NORMAL DELI,HOME HEALTH CARE,Medicare,NaN,JEHOVAH'S WITNESS,MARRIED,UNKNOWN/NOT SPECIFIED,NaN,NaN,CAROTID STENOSIS\CAROTID ANGIOGRAM AND STENT,0,1,253,M,2048-05-26,NaT,NaN,NaN,0
3,368,271,173727,249196,carevue,MICU,SICU,52,23,2120-08-07 23:12:42,2120-08-10 00:39:04,2.0600,347,2120-08-07 18:56:00,2120-08-20 16:00:00,NaT,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,HOME,Private,ENGL,NOT SPECIFIED,MARRIED,PATIENT DECLINED TO ANSWER,NaN,NaN,GALLSTONE PANCREATITIS,0,1,254,F,2074-11-30,NaT,NaN,NaN,0
4,369,272,164716,210407,carevue,CCU,CCU,57,57,2186-12-25 21:08:04,2186-12-27 12:01:13,1.6202,348,2186-12-25 21:06:00,2187-01-02 14:57:00,NaT,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,HOME,Medicare,NaN,UNOBTAINABLE,MARRIED,WHITE,NaN,NaN,PULMONARY EMBOLIS,0,1,255,M,2119-11-21,NaT,NaN,NaN,0


## Step 3: Calculate patient age
Patient age is calculated at the time of hospital admission using the year of admission and year of birth.

A year-based calculation is used instead of direct datetime subtraction to avoid overflow issues caused by date shifting in the MIMIC-III database.

In MIMIC-III, patients older than 89 are de-identified, ages above 89 were capped at 90 to account for MIMIC-III age de-identification



In [23]:
# Calculate age at admission using year difference first
cohort["AGE"] = cohort["ADMITTIME"].dt.year - cohort["DOB"].dt.year

# Adjust if birthday has not occurred yet at the time of admission
birthday_not_passed = (
    (cohort["ADMITTIME"].dt.month < cohort["DOB"].dt.month) |
    (
        (cohort["ADMITTIME"].dt.month == cohort["DOB"].dt.month) &
        (cohort["ADMITTIME"].dt.day < cohort["DOB"].dt.day)
    )
)

cohort["AGE"] = cohort["AGE"] - birthday_not_passed.astype(int)

# Fix unrealistic ages caused by de-identification
cohort["AGE"] = cohort["AGE"].apply(lambda x: 90 if (x < 0) or (x > 89) else x)

cohort[["SUBJECT_ID", "HADM_ID", "DOB", "ADMITTIME", "AGE"]].head()

,SUBJECT_ID,HADM_ID,DOB,ADMITTIME,AGE
0,268,110404,2132-02-21,2198-02-11 13:40:00,65
1,269,106296,2130-09-30,2170-11-05 11:04:00,40
2,270,188028,2048-05-26,2128-06-23 18:26:00,80
3,271,173727,2074-11-30,2120-08-07 18:56:00,45
4,272,164716,2119-11-21,2186-12-25 21:06:00,67


In [25]:
cohort["AGE"].describe()

count    61532.000000
mean        55.161688
std         26.817771
min          0.000000
25%         44.000000
50%         62.000000
75%         76.000000
max         90.000000
Name: AGE, dtype: float64

## Step 4: Keep adult patients only

Only patients aged 18 years and older were retained for the study cohort, consistent with the inclusion criteria defined in the project proposal.

In [33]:
# Keep adults only
cohort = cohort[cohort["AGE"] >= 18].copy()

print("Adult cohort shape:", cohort.shape)
print("Minimum age:", cohort["AGE"].min())

Adult cohort shape: (53330, 37)
Minimum age: 18


## Step 5: Keep the first ICU stay only
Since the proposal specifies use of the first ICU stay only, patients with multiple ICU stays are reduced to their earliest ICU admission.

In [36]:
# Sort by ICU admission time
cohort = cohort.sort_values(by=["SUBJECT_ID", "INTIME"])

# Keep first ICU stay for each patient
cohort_first_icu = cohort.drop_duplicates(subset=["SUBJECT_ID"], keep="first").copy()

print("First ICU stay cohort shape:", cohort_first_icu.shape)
cohort_first_icu[["SUBJECT_ID", "HADM_ID", "ICUSTAY_ID", "INTIME"]].head()

First ICU stay cohort shape: (38511, 37)


,SUBJECT_ID,HADM_ID,ICUSTAY_ID,INTIME
366,3,145834,211552,2101-10-20 19:10:11
367,4,185777,294638,2191-03-16 00:29:31
369,6,107064,228232,2175-05-30 21:30:54
373,9,150750,220597,2149-11-09 13:07:02
375,11,194540,229441,2178-04-16 06:19:32


## Step 6: Create sepsis labels from ICD-9 codes
Sepsis cases are identified using ICD-9 diagnosis codes listed in the project proposal:

- `99591` = Sepsis  
- `99592` = Severe Sepsis  
- `78552` = Septic Shock  
- `038xx` = Septicemia group

Patients with any of these codes will be labeled as:
- `1 = Sepsis`
- `0 = Non-Sepsis`

In [39]:
# Drop rows with missing ICD9 code
diagnoses = diagnoses.dropna(subset=["ICD9_CODE"]).copy()

# Convert ICD9_CODE to string
diagnoses["ICD9_CODE"] = diagnoses["ICD9_CODE"].astype(str).str.strip()

# Define sepsis ICD-9 codes
sepsis_exact_codes = {"99591", "99592", "78552"}

# Identify admissions with sepsis-related codes
diagnoses["SEPSIS_LABEL"] = diagnoses["ICD9_CODE"].apply(
    lambda x: 1 if (x in sepsis_exact_codes) or x.startswith("038") else 0
)

# Create admission-level sepsis label
sepsis_labels = (
    diagnoses.groupby(["SUBJECT_ID", "HADM_ID"])["SEPSIS_LABEL"]
    .max()
    .reset_index()
)

print(sepsis_labels["SEPSIS_LABEL"].value_counts())
sepsis_labels.head()

SEPSIS_LABEL
0    52566
1     6363
Name: count, dtype: int64


,SUBJECT_ID,HADM_ID,SEPSIS_LABEL
0,2,163353,0
1,3,145834,1
2,4,185777,0
3,5,178980,0
4,6,107064,0


## Step 7: Merge sepsis labels into the cohort
The admission-level sepsis labels are merged into the first-ICU-stay cohort.
Any patient without a sepsis-related diagnosis code is labeled as non-sepsis (`0`).

In [42]:
# Merge labels into cohort
cohort_final = cohort_first_icu.merge(
    sepsis_labels,
    on=["SUBJECT_ID", "HADM_ID"],
    how="left"
)

# Fill missing labels as 0
cohort_final["SEPSIS_LABEL"] = cohort_final["SEPSIS_LABEL"].fillna(0).astype(int)

print("Final cohort shape:", cohort_final.shape)
cohort_final["SEPSIS_LABEL"].value_counts()

Final cohort shape: (38511, 38)


SEPSIS_LABEL
0    34247
1     4264
Name: count, dtype: int64

## Step 8: Select key columns for the final cohort
Only the most relevant variables are kept for downstream note extraction and modeling.

In [45]:
# Keep relevant columns
cohort_final = cohort_final[
    [
        "SUBJECT_ID",
        "HADM_ID",
        "ICUSTAY_ID",
        "ADMITTIME",
        "DISCHTIME",
        "INTIME",
        "OUTTIME",
        "LOS",
        "GENDER",
        "DOB",
        "AGE",
        "DIAGNOSIS",
        "HOSPITAL_EXPIRE_FLAG",
        "SEPSIS_LABEL"
    ]
].copy()

cohort_final.head()

,SUBJECT_ID,HADM_ID,ICUSTAY_ID,ADMITTIME,DISCHTIME,INTIME,OUTTIME,LOS,GENDER,DOB,AGE,DIAGNOSIS,HOSPITAL_EXPIRE_FLAG,SEPSIS_LABEL
0,3,145834,211552,2101-10-20 19:08:00,2101-10-31 13:58:00,2101-10-20 19:10:11,2101-10-26 20:43:09,6.0646,M,2025-04-11,76,HYPOTENSION,0,1
1,4,185777,294638,2191-03-16 00:28:00,2191-03-23 18:41:00,2191-03-16 00:29:31,2191-03-17 16:46:31,1.6785,F,2143-05-12,47,"FEVER,DEHYDRATION,FAILURE TO THRIVE",0,0
2,6,107064,228232,2175-05-30 07:15:00,2175-06-15 16:00:00,2175-05-30 21:30:54,2175-06-03 13:39:54,3.6729,F,2109-06-21,65,CHRONIC RENAL FAILURE/SDA,0,0
3,9,150750,220597,2149-11-09 13:06:00,2149-11-14 10:15:00,2149-11-09 13:07:02,2149-11-14 20:52:14,5.3231,M,2108-01-26,41,HEMORRHAGIC CVA,1,0
4,11,194540,229441,2178-04-16 06:18:00,2178-05-11 19:00:00,2178-04-16 06:19:32,2178-04-17 20:21:05,1.5844,F,2128-02-22,50,BRAIN MASS,0,0


In [55]:
# Check to identify null values
cohort_final.isnull().sum()

SUBJECT_ID              0
HADM_ID                 0
ICUSTAY_ID              0
ADMITTIME               0
DISCHTIME               0
INTIME                  0
OUTTIME                 2
LOS                     2
GENDER                  0
DOB                     0
AGE                     0
DIAGNOSIS               1
HOSPITAL_EXPIRE_FLAG    0
SEPSIS_LABEL            0
dtype: int64

In [57]:
# Drop empty rows
cohort_final = cohort_final.dropna(subset=["OUTTIME", "LOS"]).copy()

# Fill Diagnosis text
cohort_final["DIAGNOSIS"] = cohort_final["DIAGNOSIS"].fillna("Unknown")

In [59]:
# Final Check
cohort_final.isnull().sum()

SUBJECT_ID              0
HADM_ID                 0
ICUSTAY_ID              0
ADMITTIME               0
DISCHTIME               0
INTIME                  0
OUTTIME                 0
LOS                     0
GENDER                  0
DOB                     0
AGE                     0
DIAGNOSIS               0
HOSPITAL_EXPIRE_FLAG    0
SEPSIS_LABEL            0
dtype: int64

## Step 9: Save the final cohort
The cleaned cohort is saved for use in the next notebook, which will focus on note extraction.

In [62]:
# Save cohort
os.makedirs(PROCESSED_PATH, exist_ok=True)
cohort_final.to_csv(PROCESSED_PATH + "sepsis_cohort.csv", index=False)

print("Cohort saved successfully.")
print("Saved file:", PROCESSED_PATH + "sepsis_cohort.csv")

Cohort saved successfully.
Saved file: ../data/processed/sepsis_cohort.csv


## Summary
In this notebook, a cleaned study cohort was created by:
- merging the core patient, admission, and ICU stay tables
- extracting age and keeping only adults
- selecting the first ICU stay per patient
- creating admission-level sepsis labels using ICD-9 codes

This cohort will be used in the next stage of the project to extract clinical notes recorded within the first 24 hours of ICU admission.